# 03 — Feature Stores

## What
A feature store is a centralised repository for computed features that serves two masters: the training pipeline (which needs historical snapshots) and the serving stack (which needs low-latency point-in-time correct values). It has two layers — an *offline store* (data warehouse / Parquet lake) for training and a *online store* (Redis / DynamoDB) for inference.

## Why
Without a feature store, teams duplicate feature logic in Python notebooks and in production services, then discover discrepancies only in production when model performance mysteriously drops. This is *training-serving skew* — the most common silent killer of production ML systems.

## Community context
Uber's Michelangelo (2017) pioneered the concept. Feast (open-source, CNCF) brought it to the broader community. Tecton, Hopsworks, and the cloud-native feature stores (Vertex, SageMaker) are now common in large organisations. Point-in-time correctness — ensuring you only use features available at the time of the label — is the central correctness concern.

In [ ]:
# Feature store with point-in-time correctness
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

class FeatureStore:
    """
    Simplified feature store demonstrating point-in-time correct lookups.
    The offline store holds (entity_id, timestamp, feature) triples.
    """
    def __init__(self):
        self.offline = {}   # feature_view -> DataFrame
        self.online  = {}   # entity_id -> {feature: value}

    def ingest(self, feature_view, df):
        """df must have: entity_id, timestamp, and feature columns"""
        df = df.copy()
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        self.offline[feature_view] = df
        # Update online store with latest values
        latest = df.sort_values('timestamp').groupby('entity_id').last()
        for eid, row in latest.iterrows():
            self.online.setdefault(eid, {})
            feat_cols = [c for c in df.columns if c not in ['entity_id','timestamp']]
            for fc in feat_cols:
                self.online[eid][fc] = row[fc]
        print(f'Ingested {len(df)} rows into {feature_view}')

    def get_historical_features(self, entity_df, feature_views):
        """
        Point-in-time correct join.
        entity_df: DataFrame with entity_id + timestamp (label event time)
        Returns: entity_df with features from each view joined correctly.
        """
        result = entity_df.copy()
        result['timestamp'] = pd.to_datetime(result['timestamp'])
        for fv in feature_views:
            store_df = self.offline[fv].sort_values('timestamp')
            feat_cols = [c for c in store_df.columns if c not in ['entity_id','timestamp']]
            joined_rows = []
            for _, row in result.iterrows():
                # Only use features computed BEFORE the label timestamp
                hist = store_df[
                    (store_df['entity_id'] == row['entity_id']) &
                    (store_df['timestamp'] <= row['timestamp'])
                ]
                if len(hist) == 0:
                    vals = {fc: np.nan for fc in feat_cols}
                else:
                    vals = hist.iloc[-1][feat_cols].to_dict()
                joined_rows.append(vals)
            for fc in feat_cols:
                result[fc] = [r[fc] for r in joined_rows]
        return result

    def get_online_features(self, entity_ids, features):
        """Low-latency lookup for serving"""
        rows = []
        for eid in entity_ids:
            row = {'entity_id': eid}
            for f in features:
                row[f] = self.online.get(eid, {}).get(f, np.nan)
            rows.append(row)
        return pd.DataFrame(rows)

# Simulate user transaction features computed over rolling 30-day windows
np.random.seed(42)
users = ['u1', 'u2', 'u3']
base_time = datetime(2024, 1, 1)

feature_rows = []
for uid in users:
    for day in range(90):
        ts = base_time + timedelta(days=day)
        feature_rows.append({
            'entity_id': uid,
            'timestamp': ts,
            'txn_count_30d': int(np.random.poisson(20 + day*0.1)),
            'avg_txn_amount_30d': round(np.random.exponential(150), 2),
        })

feature_df = pd.DataFrame(feature_rows)
store = FeatureStore()
store.ingest('user_transaction_stats', feature_df)

# Training: labels were generated at specific dates
label_events = pd.DataFrame([
    {'entity_id': 'u1', 'timestamp': '2024-02-15', 'label': 0},
    {'entity_id': 'u2', 'timestamp': '2024-03-01', 'label': 1},
    {'entity_id': 'u3', 'timestamp': '2024-02-01', 'label': 0},
])

training_data = store.get_historical_features(label_events, ['user_transaction_stats'])
print('\nPoint-in-time correct training features:')
print(training_data.to_string(index=False))

print('\nOnline features (for serving, u1):')
print(store.get_online_features(['u1'], ['txn_count_30d', 'avg_txn_amount_30d']))

## Training-Serving Skew Detection

Skew arises when features computed at training time differ from those computed at serving time — due to different code paths, different time windows, or different data sources. The fix is a *single feature definition* shared by both paths, exactly what the feature store enforces.

In [ ]:
# Detect training-serving skew by comparing distributions
from scipy import stats

def detect_skew(training_vals, serving_vals, feature_name, alpha=0.05):
    stat, p = stats.ks_2samp(training_vals, serving_vals)
    skewed = p < alpha
    print(f'{feature_name}: KS={stat:.3f} p={p:.4f} -> {"SKEW DETECTED" if skewed else "OK"}')
    return skewed

# Simulate slight bug: serving pipeline uses 7-day window instead of 30-day
train_counts = np.random.poisson(20, 1000)  # 30-day window
serve_counts = np.random.poisson(5, 200)    # accidentally 7-day window
detect_skew(train_counts, serve_counts, 'txn_count')

# Correct version: same window in both paths
serve_counts_fixed = np.random.poisson(20, 200)
detect_skew(train_counts, serve_counts_fixed, 'txn_count_fixed')